In [1]:
!pwd

/home/richard/source/madb_data/experiments


In [2]:
import sys
from pathlib import Path

# Add the repo root to sys.path
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from thesis_rec import thesis_rec

In [3]:
def load_all_records():
    from pathlib import Path
    import json
    from thesis_rec import thesis_rec  # Adjust if needed

    base_path = Path("..") / "thesis_data"
    all_records = []

    for subdir in base_path.glob("*/madb"):
        for json_file in subdir.glob("ENSG*.json"):
            try:
                with open(json_file) as f:
                    outer = json.load(f)
                    rich_dict_str = outer.get("data")
                    if rich_dict_str is None:
                        raise ValueError(f"No 'data' field in {json_file}")
                    rich_dict = json.loads(rich_dict_str)
                    record = thesis_rec.from_rich_dict(rich_dict)
                    all_records.append(record)
            except Exception as e:
                print(f"Failed to load {json_file}: {e}")

    
    return all_records

records = load_all_records()
print(f"Loaded {len(records)} records")

Loaded 42 records


In [16]:
def extract_lengths(records, kmer_size=20):
    sw_lengths = []
    braid_lengths = []
    ids = []

    for rec in records:
        sw_len = rec.madb_ungapped_smith_waterman_length
        braid_data = rec.madb_longest_braid_length
        if braid_data and str(kmer_size) in braid_data:
            braid_len = braid_data['25']
        if (
            sw_len is not None and
            isinstance(braid_data, dict) and
            braid_data['25'] is not None
        ):
            braid_len = braid_data['25']
            sw_lengths.append(sw_len)
            braid_lengths.append(braid_len)
            ids.append(rec.unique_id)

    return sw_lengths, braid_lengths, ids


In [15]:
import plotly.express as px
import pandas as pd

records = load_all_records()
sw_lengths, braid_lengths, ids = extract_lengths(records)

df = pd.DataFrame({
    "SW Length": sw_lengths,
    "Longest Braid Length": braid_lengths,
    "ID": ids
})

fig = px.scatter(df, x="SW Length", y="Longest Braid Length", hover_name="ID",
                 title="Ungapped Smith-Waterman vs Longest Braid Length",
                 labels={"SW Length": "SW Ungapped Length", "Longest Braid Length": "Longest Braid Length"})

# Add parity line (y=x)
fig.add_shape(
    type="line",
    x0=0, y0=0,
    x1=max(df["SW Length"].max(), df["Longest Braid Length"].max()),
    y1=max(df["SW Length"].max(), df["Longest Braid Length"].max()),
    line=dict(dash="dash", color="gray")
)

fig.show()
